# 실험 결과 분석

모델 × 데이터셋 × 전략 3축 비교  
- **데이터**: `summary.json`, `trajectory_logs.jsonl`, `step_logs.jsonl`
- **비교 축**: 모델(phi35mini / qwen7bcoder), 데이터셋(humaneval / mbpp), 전략(single / repair / code_then_plan / code_then_plan_repair / policy_loop 등)

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

## 0. 경로 설정

In [ ]:
RESULTS_ROOT = Path("../results")

# 존재하는 결과 폴더 자동 탐색
def discover_runs(results_root: Path) -> list[dict]:
    """
    results/<model>/<dataset>/<strategy>/summary.json 구조를 재귀 탐색.
    summary.json 이 있는 폴더를 하나의 run으로 간주.
    """
    runs = []
    for summary_path in sorted(results_root.rglob("summary.json")):
        parts = summary_path.parent.relative_to(results_root).parts
        if len(parts) >= 3:
            model, dataset, strategy = parts[0], parts[1], parts[2]
        elif len(parts) == 2:
            model, dataset, strategy = parts[0], parts[1], "unknown"
        else:
            continue
        runs.append({
            "model": model,
            "dataset": dataset,
            "strategy": strategy,
            "summary_path": summary_path,
            "dir": summary_path.parent,
        })
    return runs

runs = discover_runs(RESULTS_ROOT)
print(f"발견된 run 수: {len(runs)}")
for r in runs:
    print(f"  {r['model']:20s} | {r['dataset']:15s} | {r['strategy']}")

## 1. summary.json 로드 → 핵심 지표 비교표

In [ ]:
def load_summary(run: dict) -> dict:
    with open(run["summary_path"], encoding="utf-8") as f:
        data = json.load(f)

    row = {
        "model":    run["model"],
        "dataset":  run["dataset"],
        "strategy": run["strategy"],
    }

    # 공통 지표
    row["total_problems"]         = data.get("total_problems", None)
    row["num_success"]            = data.get("num_success", None)
    row["success_at_k"]           = data.get("success_at_k", None)
    row["execution_success_rate"] = data.get("execution_success_rate", None)
    row["conditional_success"]    = data.get("conditional_success", None)
    row["max_calls"]              = data.get("max_calls", None)

    # extra_summary (failure breakdown)
    extra = data.get("extra_summary", {})
    row["code_failed"]         = extra.get("code_failed", None)
    row["define_test_failed"]  = extra.get("define_test_failed", None)
    row["run_test_failed"]     = extra.get("run_test_failed", None)

    # planning stats (code_then_plan_repair 계열)
    ps = data.get("planning_stats", {})
    row["plan_used"]              = ps.get("used_plan", None)
    row["repair_used"]            = ps.get("used_repair", None)
    row["planning_recovery_rate"] = ps.get("planning_recovery_rate", None)
    row["repair_recovery_rate"]   = ps.get("repair_recovery_rate", None)

    # policy_stats (policy_loop 계열)
    pol = data.get("policy_stats", {})
    pl = pol.get("problem_level", {})
    cl = pol.get("call_level", {})
    if pl:
        row["plan_used"]              = pl.get("plan_used_problems", row.get("plan_used"))
        row["repair_used"]            = pl.get("repair_used_problems", row.get("repair_used"))
        row["planning_recovery_rate"] = pl.get("plan_problem_recovery_rate", row.get("planning_recovery_rate"))
        row["repair_recovery_rate"]   = pl.get("repair_problem_recovery_rate", row.get("repair_recovery_rate"))
    if cl:
        row["planning_cycle_count"]        = cl.get("planning_cycle_count", None)
        row["planning_cycle_success_rate"] = cl.get("planning_cycle_success_rate", None)
        row["repair_call_count"]           = cl.get("repair_call_count", None)
        row["repair_call_success_rate"]    = cl.get("repair_call_success_rate", None)

    return row


summary_rows = [load_summary(r) for r in runs]
df_summary = pd.DataFrame(summary_rows)
df_summary

### 1-1. 핵심 지표만 추린 비교표

In [ ]:
KEY_COLS = [
    "model", "dataset", "strategy",
    "total_problems", "num_success",
    "success_at_k", "execution_success_rate", "conditional_success",
]
df_summary[KEY_COLS].sort_values(["dataset", "model", "strategy"])

### 1-2. 전략별 피벗 (dataset × model 고정, 전략 비교)

In [ ]:
for dataset in df_summary["dataset"].unique():
    sub = df_summary[df_summary["dataset"] == dataset]
    pivot = sub.pivot_table(
        index="model",
        columns="strategy",
        values="success_at_k",
    )
    print(f"\n=== {dataset} | success_at_k ===")
    display(pivot)

### 1-3. 모델별 피벗 (strategy × dataset)

In [ ]:
for model in df_summary["model"].unique():
    sub = df_summary[df_summary["model"] == model]
    pivot = sub.pivot_table(
        index="strategy",
        columns="dataset",
        values=["success_at_k", "execution_success_rate", "conditional_success"],
    )
    print(f"\n=== {model} ===")
    display(pivot)

## 2. trajectory_logs.jsonl 로드 → 문제별 분석

In [ ]:
def load_jsonl(path: Path) -> list[dict]:
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def load_trajectory_logs(runs: list[dict]) -> pd.DataFrame:
    frames = []
    for r in runs:
        p = r["dir"] / "trajectory_logs.jsonl"
        if not p.exists():
            continue
        rows = load_jsonl(p)
        df = pd.DataFrame(rows)
        df["model"]    = r["model"]
        df["dataset"]  = r["dataset"]
        df["strategy"] = r["strategy"]
        frames.append(df)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


df_traj = load_trajectory_logs(runs)
print(f"trajectory rows: {len(df_traj)}")
df_traj.head(3)

### 2-1. call_count 분포 (전략별 평균 호출 수)

In [ ]:
if not df_traj.empty and "call_count" in df_traj.columns:
    display(
        df_traj.groupby(["model", "dataset", "strategy"])["call_count"]
        .agg(["mean", "median", "min", "max", "count"])
        .round(2)
        .sort_index()
    )

### 2-2. 문제별 최종 상태 분포

In [ ]:
if not df_traj.empty and "final_status" in df_traj.columns:
    status_col = "final_status"
    # coarse failure family
    df_traj["failure_family_coarse"] = df_traj[status_col].apply(
        lambda s: "PASS" if s == "PASS" else str(s).split(":")[0]
    )
    display(
        df_traj.groupby(["model", "dataset", "strategy", "failure_family_coarse"])
        .size()
        .rename("count")
        .reset_index()
        .sort_values(["model", "dataset", "strategy", "count"], ascending=[True, True, True, False])
    )

### 2-3. 복구 전략 효과 (recovered_by 분포)

In [ ]:
if not df_traj.empty and "recovered_by" in df_traj.columns:
    display(
        df_traj.groupby(["model", "dataset", "strategy", "recovered_by"])
        .size()
        .rename("count")
        .reset_index()
        .sort_values(["model", "dataset", "strategy", "count"], ascending=[True, True, True, False])
    )

### 2-4. 초기 실패 → 최종 성공 전환율 (recovery rate)

In [ ]:
if not df_traj.empty:
    # initial_failed 컬럼이 없으면 generate step 기준으로 직접 계산
    if "initial_failed" not in df_traj.columns:
        df_traj["initial_failed"] = df_traj.get("transition_path", pd.Series()).apply(
            lambda p: (p[0].split(":")[0] != "PASS") if isinstance(p, list) and p else None
        )

    if "final_status" in df_traj.columns:
        df_traj["final_passed"] = df_traj["final_status"] == "PASS"

    recovery = (
        df_traj[df_traj["initial_failed"] == True]
        .groupby(["model", "dataset", "strategy"])["final_passed"]
        .agg(recovered="sum", total="count")
    )
    recovery["recovery_rate"] = recovery["recovered"] / recovery["total"]
    display(recovery.round(4))

## 3. step_logs.jsonl 로드 → call 수준 분석

In [ ]:
def load_step_logs(runs: list[dict]) -> pd.DataFrame:
    frames = []
    for r in runs:
        p = r["dir"] / "step_logs.jsonl"
        if not p.exists():
            continue
        rows = load_jsonl(p)
        df = pd.DataFrame(rows)
        df["model"]    = r["model"]
        df["dataset"]  = r["dataset"]
        df["strategy"] = r["strategy"]
        frames.append(df)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


df_steps = load_step_logs(runs)
print(f"step rows: {len(df_steps)}")
df_steps.head(3)

### 3-1. stage별 PASS 비율 (call 수준 성공률)

In [ ]:
if not df_steps.empty and "stage" in df_steps.columns and "status" in df_steps.columns:
    df_steps["passed"] = df_steps["status"] == "PASS"
    stage_stats = (
        df_steps[df_steps["stage"].isin(["generate", "repair", "plan_code"])]
        .groupby(["model", "dataset", "strategy", "stage"])["passed"]
        .agg(pass_count="sum", total="count")
    )
    stage_stats["pass_rate"] = stage_stats["pass_count"] / stage_stats["total"]
    display(stage_stats.round(4))

### 3-2. error_type 분포 (실패 원인 분석)

In [ ]:
if not df_steps.empty and "error_type" in df_steps.columns:
    failed_steps = df_steps[
        df_steps["status"].notna()
        & (df_steps["status"] != "PASS")
        & (df_steps["status"] != "PLAN_DONE")
    ]
    display(
        failed_steps.groupby(["model", "dataset", "strategy", "error_type"])
        .size()
        .rename("count")
        .reset_index()
        .sort_values(["model", "dataset", "strategy", "count"], ascending=[True, True, True, False])
        .head(60)
    )

### 3-3. 토큰 사용량 (step 기준)

In [ ]:
if not df_steps.empty and "total_tokens" in df_steps.columns:
    display(
        df_steps.groupby(["model", "dataset", "strategy", "stage"])["total_tokens"]
        .agg(["mean", "median", "sum"])
        .round(1)
        .sort_index()
    )

## 4. 전체 성능 요약 테이블 (논문/보고서용)

In [ ]:
REPORT_COLS = [
    "model", "dataset", "strategy",
    "success_at_k", "execution_success_rate", "conditional_success",
    "code_failed", "define_test_failed", "run_test_failed",
]
existing = [c for c in REPORT_COLS if c in df_summary.columns]
df_report = (
    df_summary[existing]
    .sort_values(["dataset", "model", "success_at_k"], ascending=[True, True, False])
    .reset_index(drop=True)
)
display(df_report)

In [ ]:
# CSV로 저장
out_path = Path("results_report.csv")
df_report.to_csv(out_path, index=False)
print(f"저장 완료: {out_path.resolve()}")